In [ ]:
import rmllm
import json
import pandas as pd

In [ ]:
data_dir = rmllm.config.EXTERNAL_DATA_DIR
models = ["gemma3:12b","gemma3:12b-it-qat","gemma3:27b","gemma3:27b-it-qat","llama3.3:70b","llama4:scout"]
tasks = ["rm_2op","rm_2op_r","rm_2op_tc","rm_2op_tc_r"]
memories = ["SingleTurn","Convo"] 

In [ ]:
def op2(data):
    x = data["taskdata"]
    y = []
    for i in x:
        #print(i)
        y.append(
            {
                "tridx": i["trial_idx"],
                "trcode": i["trcode"],
                "taskname": i["taskname"],
                "source": i["context_item"],
                "trace": i["trace_id"],
                **(eval(i["pred_resp"]["content"])),
                **(i["stimulus"]["Word_Pair"])
            }
        )
    return y

def op2_tc(data):
    x = data["taskdata"]
    y = []
    for i in x:
        j = {
            "tridx": i["trial_idx"],
            "trcode": i["trcode"],
            "taskname": i["taskname"],
            "source": i["context_item"],
            "trace": i["trace_id"],
            **((eval(i["pred_resp"]["content"]))["response"]),
        }
        if "Word_Pair" in i["stimulus"]:
            j["word_1"] = i["stimulus"]["Word_Pair"]["word_1"]
            j["word_2"] = i["stimulus"]["Word_Pair"]["word_2"]

        y.append(j)

    return y


In [ ]:
def organize_tc_2op(data):
    all_trials = []
    for i in data["trcode"].unique():
        trial_i = data.loc[data["trcode"] == i]
        all_trials.append({
            "Word_2": trial_i["Word_2"].dropna().values[0],
            "word_1": trial_i["word_1"].dropna().values[0],
            "word_2": trial_i["word_2"].dropna().values[0],
            "Rating": trial_i["Relatedness_Rating"].dropna().values[0],
            "Judgment": trial_i["Judgment"].dropna().values[0],
            "Confidence": trial_i["Confidence"].dropna().values[0],
            "taskname": trial_i["taskname"].unique()[0],
            "source": trial_i["source"].unique()[0],
            "trace": trial_i["trace"].unique()[0],
            "trcode": trial_i["trcode"].unique()[0],
            "model": trial_i["model"].unique()[0],
            "memory": trial_i["memory"].unique()[0],
            "task":trial_i["task"].unique()[0]
        })
    return pd.DataFrame(all_trials)


In [ ]:
y = {}
z = pd.DataFrame()
for model in models:
    y[model] = {}
    for memory in memories:
        y[model][memory] = {}
        for task in tasks:
            
            task_dir = data_dir/task/"RM"/f"zs_{task}"/f"ollama_{model}_{memory}" 
            print(task_dir.is_dir())
            if task_dir.is_dir():
                for i in task_dir.iterdir():
                    if "0-" in i.name:
                        with open(i,"r") as f:
                            taskdict = json.load(f)
                            print(taskdict)
                            if ("2op" in task) & ("tc" not in task):
                                y[model][memory][task] = op2(taskdict)
                            if "2op_tc" in task:
                                y[model][memory][task] = op2_tc(taskdict)
                            
                            y[model][memory][task] = pd.DataFrame(y[model][memory][task])
                            y[model][memory][task]["model"] = model
                            y[model][memory][task]["task"] = task
                            y[model][memory][task]["memory"] = memory
                            
                            if "2op_tc" in task:
                                y[model][memory][task] = organize_tc_2op(
                                    y[model][memory][task]
                                )
                            
                            
                            
                            z = pd.concat([z, y[model][memory][task]])

task_dir

In [ ]:
task_map = {
    "rm_2op": "task_1",
    "rm_2op_r": "task_1",
    "rm_2op_tc":"task_2",
    "rm_2op_tc_r":"task_2",

}
exp_map = {
    "task_1":1,
    "task_2":1, 

}

order_map = {
    "rm_2op": 1,
    "rm_2op_r": 2,
    "rm_2op_tc": 1, # external first, internal second
    "rm_2op_tc_r": 2, # internal first, external second

}

In [ ]:
z["experiment"] = z["task"].map(task_map)
z["study"] = z["experiment"].map(exp_map)
z["order"] = z["task"].map(order_map)

In [ ]:
exp_1 = z.loc[z["study"] == 1]


# 1) Experiment 1 : 2 options 

In [ ]:
exp1_corr = {
    "imagined": "internal",
    "perceived": "external"
}
exp_1["corrAns"] = exp_1["source"].map(exp1_corr)

exp_1_perceived = exp_1.loc[exp_1["source"]=="perceived"]
exp_1_imagined = exp_1.loc[exp_1["source"] == "imagined"]
exp_1_perceived["word2acc"] = exp_1_perceived["Word_2"] == exp_1_perceived["word_2"]
exp_1_imagined["word2acc"] = exp_1_imagined["Word_2"] != exp_1_imagined["word_1"]

In [ ]:
exp_1 = pd.concat([exp_1_perceived, exp_1_imagined])
exp_1_incorrect_trials = exp_1.loc[exp_1["word2acc"] == False]
dropouts = exp_1_incorrect_trials.groupby(["experiment","model","source"],as_index=False).size()
dropouts["proportion"] = dropouts["size"]/144
dropouts

In [ ]:
import seaborn as sns
sns.catplot(x="model", hue="experiment", y="proportion",kind="bar", data=dropouts)

In [ ]:
exp_1.loc[:, "accuracy"] = exp_1["Judgment"] == exp_1["corrAns"]

# SAVE DATA

In [ ]:
out_dir = rmllm.config.PROCESSED_DATA_DIR
exp_1.to_csv(out_dir/"rmai_exp_1.csv")